## X, y and train-test-split

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.db import get_engine, fetch_table
from src.config import FEATURE_COLS, FEATURE_COLS_NO_WIFI
from src.config import MODEL_TEXT_COLS, MODEL_NUMBER_COLS, MODEL_RATING_COLS
from src.features import make_features

print("Project root:", project_root)

Project root: d:\OMEN-16\ML-Projects\airline-satisfaction-ml


In [2]:
engine = get_engine()
df = fetch_table(engine)

print("Shape:", df.shape)
print("First id:", df["id"].iloc[0], "| last id:", df["id"].iloc[-1])

Shape: (129880, 24)
First id: 1 | last id: 129880


## Building X

In [3]:
# X holds the columns the model learns from
X = make_features(df)

# the same table without wifi
X_no_wifi = X[FEATURE_COLS_NO_WIFI]

print("X shape        :", X.shape)
print("X_no_wifi shape:", X_no_wifi.shape)

X shape        : (129880, 18)
X_no_wifi shape: (129880, 17)


In [4]:
X.head()

,customer_type,type_of_travel,travel_class,age,flight_distance,departure_delay,ease_of_online_booking,checkin_service,online_boarding,onboard_service,seat_comfort,leg_room_service,cleanliness,food_and_drink,inflight_service,inflight_wifi_service,inflight_entertainment,baggage_handling
0,First-time,Business,Business,48,821,2,3,4,3,3,5,2,5,5,5,3,5,5
1,Returning,Business,Business,35,821,26,2,3,5,5,4,5,5,3,5,2,5,5
2,Returning,Business,Business,41,853,0,4,4,5,3,5,3,5,5,3,4,3,3
3,Returning,Business,Business,50,1905,0,2,3,4,5,5,5,4,4,5,2,5,5
4,Returning,Business,Business,49,3470,0,3,3,5,3,4,4,5,4,3,3,3,3


#### Building y

In [5]:
# y holds the answer we want to predict: 1 = satisfied, 0 = not satisfied
y = (df["satisfaction"] == "Satisfied").astype(int)
y.name = "satisfied"

print("y shape:", y.shape)
print()
print(y.value_counts())
print()
print("Share satisfied:", round(y.mean(), 4))

y shape: (129880,)

satisfied
0    73452
1    56428
Name: count, dtype: int64

Share satisfied: 0.4345


In [6]:
# X and y side by side, first 5 passengers
pd.concat([X.head(), y.head()], axis=1)

,customer_type,type_of_travel,travel_class,age,flight_distance,departure_delay,ease_of_online_booking,checkin_service,online_boarding,onboard_service,seat_comfort,leg_room_service,cleanliness,food_and_drink,inflight_service,inflight_wifi_service,inflight_entertainment,baggage_handling,satisfied
0,First-time,Business,Business,48,821,2,3,4,3,3,5,2,5,5,5,3,5,5,0
1,Returning,Business,Business,35,821,26,2,3,5,5,4,5,5,3,5,2,5,5,1
2,Returning,Business,Business,41,853,0,4,4,5,3,5,3,5,5,3,4,3,3,1
3,Returning,Business,Business,50,1905,0,2,3,4,5,5,5,4,4,5,2,5,5,1
4,Returning,Business,Business,49,3470,0,3,3,5,3,4,4,5,4,3,3,3,3,1


In [7]:
print("Rows in X:", len(X))
print("Rows in y:", len(y))
print("Same number of rows       :", len(X) == len(y))
print("Row numbers line up       :", X.index.equals(y.index))

Rows in X: 129880
Rows in y: 129880
Same number of rows       : True
Row numbers line up       : True


#### checking is anthing in X that shouldn't be

In [8]:
print("'satisfaction' is in X:", "satisfaction" in X.columns)
print("'satisfied' is in X   :", "satisfied" in X.columns)
print("'id' is in X          :", "id" in X.columns)
print()
print("df still has", df.shape[1], "columns and no 'satisfied' column:", "satisfied" not in df.columns)

'satisfaction' is in X: False
'satisfied' is in X   : False
'id' is in X          : False

df still has 24 columns and no 'satisfied' column: True


#### checking blanks and data types

In [9]:
print("Missing values in X:", X.isnull().sum().sum())
print("Missing values in y:", y.isnull().sum())
print()
print(X.dtypes.value_counts())

Missing values in X: 0
Missing values in y: 0

int64    15
str       3
Name: count, dtype: int64


In [10]:
number_cols = MODEL_NUMBER_COLS + MODEL_RATING_COLS

corr = X[number_cols].corrwith(y).abs().sort_values(ascending=False)

print("Three highest correlations with y:")
print(corr.head(3).round(3))

Three highest correlations with y:
online_boarding           0.502
inflight_entertainment    0.398
seat_comfort              0.349
dtype: float64


observation : 
- here the strongest link is 0.50, the same online boarding figure from Step 3. That's what a healthy X looks like. If a column were a copy of y, or gave the answer away, its correlation would be close to 1.0.

- this check can only catch obvious leaks. It's a smoke alarm, not a guarantee. The real protection is rule 4 from 6.5.4: knowing where each column comes from.

In [11]:
# what a leak would look like: a column that is a copy of y
leak_test = X[["age"]].copy()
leak_test["copy_of_y"] = y

print(leak_test.corrwith(y).round(3))

age          0.134
copy_of_y    1.000
dtype: float64


#### what is inside of X

In [12]:
# what a leak would look like: a column that is a copy of y
leak_test = X[["age"]].copy()
leak_test["copy_of_y"] = y

print(leak_test.corrwith(y).round(3))

age          0.134
copy_of_y    1.000
dtype: float64


#### the label ineach text column

In [13]:

for col in MODEL_TEXT_COLS:
    print(col, "->", sorted(X[col].unique()))

customer_type -> ['First-time', 'Returning']
type_of_travel -> ['Business', 'Personal']
travel_class -> ['Business', 'Economy', 'Economy Plus']


#### the Range of every number column

In [14]:
X[MODEL_NUMBER_COLS + MODEL_RATING_COLS].describe().T[["count", "mean", "min", "max"]].round(2)

,count,mean,min,max
age,129880.0,39.43,7.0,85.0
flight_distance,129880.0,1190.32,31.0,4983.0
departure_delay,129880.0,14.71,0.0,1592.0
ease_of_online_booking,129880.0,2.76,0.0,5.0
checkin_service,129880.0,3.31,0.0,5.0
online_boarding,129880.0,3.25,0.0,5.0
onboard_service,129880.0,3.38,0.0,5.0
seat_comfort,129880.0,3.44,0.0,5.0
leg_room_service,129880.0,3.35,0.0,5.0
cleanliness,129880.0,3.29,0.0,5.0


#### The one column that seperates column X from X_no_wifi

In [15]:
print("Only in X         :", set(X.columns) - set(X_no_wifi.columns))
print("Only in X_no_wifi :", set(X_no_wifi.columns) - set(X.columns))

Only in X         : {'inflight_wifi_service'}
Only in X_no_wifi : set()


## Spliting the data
#### Spliting x and y into training(80%) and test rows(20%)

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (103904, 18)
X_test : (25976, 18)
y_train: (103904,)
y_test : (25976,)


In [17]:
total = len(X)

print("Total rows  :", total)
print("Train rows  :", len(X_train), "->", round(len(X_train) / total * 100, 1), "%")
print("Test rows   :", len(X_test), "->", round(len(X_test) / total * 100, 1), "%")
print("Train + test:", len(X_train) + len(X_test))

Total rows  : 129880
Train rows  : 103904 -> 80.0 %
Test rows   : 25976 -> 20.0 %
Train + test: 129880


In [18]:
print("Share satisfied in all the data:", round(y.mean(), 4))
print("Share satisfied in y_train      :", round(y_train.mean(), 4))
print("Share satisfied in y_test       :", round(y_test.mean(), 4))
print()
print("y_train counts:")
print(y_train.value_counts())
print()
print("y_test counts:")
print(y_test.value_counts())

Share satisfied in all the data: 0.4345
Share satisfied in y_train      : 0.4345
Share satisfied in y_test       : 0.4345

y_train counts:
satisfied
0    58762
1    45142
Name: count, dtype: int64

y_test counts:
satisfied
0    14690
1    11286
Name: count, dtype: int64


#### the no_wifi version

In [19]:
X_train_no_wifi = X_train[FEATURE_COLS_NO_WIFI]
X_test_no_wifi = X_test[FEATURE_COLS_NO_WIFI]

print("X_train_no_wifi:", X_train_no_wifi.shape)
print("X_test_no_wifi :", X_test_no_wifi.shape)
print("Same passengers as X_train:", X_train_no_wifi.index.equals(X_train.index))
print("Same passengers as X_test :", X_test_no_wifi.index.equals(X_test.index))

X_train_no_wifi: (103904, 17)
X_test_no_wifi : (25976, 17)
Same passengers as X_train: True
Same passengers as X_test : True


NOTE : 

- Here in this  instead of splitting a second time, i taken the same passengers from X_train and X_test and keep only the 17 no-wifi columns. That guarantees the with-wifi and without-wifi experiments use exactly the same passengers, so any difference in score comes from wifi alone, not from a different split.

#### checking and looking inside

In [20]:
overlap = set(X_train.index) & set(X_test.index)

print("Passengers in both train and test:", len(overlap))
print("Passengers in train or test      :", len(set(X_train.index) | set(X_test.index)))
print()
print("X_train and y_train line up:", X_train.index.equals(y_train.index))
print("X_test and y_test line up  :", X_test.index.equals(y_test.index))

Passengers in both train and test: 0
Passengers in train or test      : 129880

X_train and y_train line up: True
X_test and y_test line up  : True


In [21]:
print("First 5 row numbers in X_train:", list(X_train.index[:5]))
print("First 5 row numbers in X_test :", list(X_test.index[:5]))

X_train.head()

First 5 row numbers in X_train: [57644, 103332, 76605, 121595, 107087]
First 5 row numbers in X_test : [126481, 12424, 128951, 2489, 44988]


,customer_type,type_of_travel,travel_class,age,flight_distance,departure_delay,ease_of_online_booking,checkin_service,online_boarding,onboard_service,seat_comfort,leg_room_service,cleanliness,food_and_drink,inflight_service,inflight_wifi_service,inflight_entertainment,baggage_handling
57644,Returning,Business,Business,56,1848,0,4,3,2,4,5,4,1,4,4,4,4,4
103332,Returning,Personal,Economy,70,1371,3,3,3,4,1,5,3,4,4,1,3,1,1
76605,Returning,Business,Business,48,2473,0,1,5,4,5,5,5,5,4,5,3,5,5
121595,Returning,Personal,Economy,63,912,40,3,2,3,4,5,4,5,5,2,3,5,2
107087,Returning,Business,Business,57,562,12,4,3,4,4,4,4,3,4,4,4,4,4


Note :
- as we can see here the row numbers are jumbled, not 0, 1, 2. The split shuffled the passengers.

- but They are the original row numbers from before the split. Passenger 57,644 in X_train is the same person as row 57,644 in df. Keeping the original numbers on purpose is how X and y stay matched.

#### checking how does train and test look alike

In [22]:
print("Class mix in train (%):")
print((X_train["travel_class"].value_counts(normalize=True) * 100).round(1))
print()
print("Class mix in test (%):")
print((X_test["travel_class"].value_counts(normalize=True) * 100).round(1))

Class mix in train (%):
travel_class
Business        47.8
Economy         44.9
Economy Plus     7.3
Name: proportion, dtype: float64

Class mix in test (%):
travel_class
Business        48.0
Economy         45.0
Economy Plus     7.0
Name: proportion, dtype: float64


NOTE :
- As we can see here that The two mixes are very close. They aren't identical, because we had only stratified only on the answer (y), not on class. A 0.3-point gap in Economy Plus is ordinary random variation.

#### here are few more comparision

In [23]:
print("Average age, train          :", round(X_train["age"].mean(), 2))
print("Average age, test           :", round(X_test["age"].mean(), 2))
print()
print("Average distance, train     :", round(X_train["flight_distance"].mean(), 1))
print("Average distance, test      :", round(X_test["flight_distance"].mean(), 1))
print()
print("Personal travel (%), train  :", round((X_train["type_of_travel"] == "Personal").mean() * 100, 1))
print("Personal travel (%), test   :", round((X_test["type_of_travel"] == "Personal").mean() * 100, 1))


Average age, train          : 39.4
Average age, test           : 39.54

Average distance, train     : 1190.5
Average distance, test      : 1189.6

Personal travel (%), train  : 31.0
Personal travel (%), test   : 30.8


NOTE :
- Actually Average age differs by 0.14 years, average distance by under 1 mile, and the personal-travel share by 0.2 points. The test set is a faithful small copy of the training set. If one of these differed a lot, something would be wrong with the split.

In [28]:
# rebuild the split exactly the way we did it in feature_engineering
data = df.drop(columns="id")
data["satisfied"] = y

feature_engineering_train, feature_engineering_test = train_test_split(data, test_size=0.2, random_state=42, stratify=data["satisfied"])

print("Same test rows as feature engineering :", sorted(feature_engineering_test.index) == sorted(X_test.index))
print("Same train rows as feature engineering :", sorted(feature_engineering_train.index) == sorted(X_train.index))

Same test rows as feature engineering : True
Same train rows as feature engineering : True
